<a href="https://colab.research.google.com/github/Nihat0311/NLP/blob/main/Knowledge_Distillation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
import torch.nn.functional as F
device='cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [ ]:
dataset=load_dataset('LocalDoc/summarization_azerbaijan')

full_train=dataset['train']
small_data=full_train.select(range(min(5000,len(full_train))))

small_data

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/207M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/115605 [00:00<?, ? examples/s]

Dataset({
    features: ['index', 'text', 'summary'],
    num_rows: 5000
})

In [ ]:
def clean_function(example):
  text=example['text']
  summary=example['summary']

  if text is None or summary is None:
    return False

  if len(text.strip())<50:
    return False
  if len(summary.strip())<5:
    return False
  return True

small_data=small_data.filter(clean_function)
small_data

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dataset({
    features: ['index', 'text', 'summary'],
    num_rows: 5000
})

In [ ]:
teacher_name = "nijatzeynalov/mT5-based-azerbaijani-summarize"

teacher_tokenizer=AutoTokenizer.from_pretrained(teacher_name)
teacher_model=AutoModelForSeq2SeqLM.from_pretrained(teacher_name).to(device)

teacher_model.eval()

config.json:   0%|          | 0.00/867 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/455 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/191 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

MT5ForConditionalGeneration(
  (shared): Embedding(250112, 512)
  (encoder): MT5Stack(
    (embed_tokens): Embedding(250112, 512)
    (block): ModuleList(
      (0): MT5Block(
        (layer): ModuleList(
          (0): MT5LayerSelfAttention(
            (SelfAttention): MT5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): MT5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): MT5LayerFF(
            (DenseReluDense): MT5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
          

In [ ]:
def teacher_summarize(text):
  inputs=teacher_tokenizer(
      text,
      return_tensors='pt',
      padding='max_length',
      truncation=True,
      max_length=1024
  ).to(device)

  with torch.no_grad():
    summary_ids=teacher_model.generate(
        **inputs,
        num_beams=4,
        max_length=128,
        min_length=15,
        no_repeat_ngram_size=3,
        early_stopping=True
    )
  return teacher_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [ ]:
def add_teacher_summary_batch(batch):
  texts=batch['text']
  inputs=teacher_tokenizer(
      texts,
      return_tensors='pt',
      padding=True,
      truncation=True,
      max_length=512
  ).to(device)

  with torch.inference_mode():
    summary_ids=teacher_model.generate(
        **inputs,
        num_beams=2,
        max_length=80,
        min_length=10,
        no_repeat_ngram_size=3,
        early_stopping=True
    )
  summaries=teacher_tokenizer.batch_decode(summary_ids, skip_special_tokens=True)
  batch['teacher_summary']=summaries
  return batch

In [ ]:
distill_data=small_data.map(
    add_teacher_summary_batch,
    batched=True,
    batch_size=32
)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

student_name='google/mt5-small'

student_tokenizer=AutoTokenizer.from_pretrained(student_name)
student_model=AutoModelForSeq2SeqLM.from_pretrained(student_name)

student_model=student_model.to(device)
student_model.config.use_cache=False

print(next(student_model.parameters()).dtype)
print(next(student_model.parameters()).device)

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


torch.float32
cuda:0


In [ ]:
dataset_split=distill_data.train_test_split(test_size=0.05,seed=42)
dataset_split

DatasetDict({
    train: Dataset({
        features: ['index', 'text', 'summary', 'teacher_summary'],
        num_rows: 4750
    })
    test: Dataset({
        features: ['index', 'text', 'summary', 'teacher_summary'],
        num_rows: 250
    })
})

In [ ]:
max_input_length=512
max_target_length=128

prefix='summarize: '

def preprocess_function(examples):
  inputs=[prefix+text
          for text in examples['text']]
  targets=examples['teacher_summary']

  model_inputs=student_tokenizer(
      inputs,
      max_length=max_input_length,
      truncation=True
  )

  labels=student_tokenizer(
      text_target=targets,
      max_length=max_target_length,
      truncation=True
  )
  model_inputs['labels']=labels['input_ids']
  return model_inputs

In [ ]:
tokenized_dataset=dataset_split.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset_split['train'].column_names
)

Map:   0%|          | 0/4750 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

In [ ]:
data_collator=DataCollatorForSeq2Seq(
    tokenizer=student_tokenizer,
    model=student_model
)

In [ ]:
training_args=Seq2SeqTrainingArguments(
    output_dir="./distilled-mt5-az-summary",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,

    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,

    save_strategy='steps',
    save_steps=500,
    save_total_limit=2,

    predict_with_generate=False,
    fp16=False,
    bf16=False,

    max_grad_norm=1.0,
    )

In [ ]:
class DistillationSeq2SeqTrainer(Seq2SeqTrainer):
  def __init__(self,teacher_model,temperature=2.0,alpha=0.7,*args,**kwargs):
    super().__init__(*args, **kwargs)

    self.teacher_model = teacher_model
    self.teacher_model.eval()

    for p in self.teacher_model.parameters():
      p.requires_grad=False

      self.temperature = temperature
      self.alpha = alpha
  def compute_loss(self, model, inputs, return_outputs=False,**kwargs):
    labels = inputs["labels"]

    student_outputs=model(**inputs)
    student_logits=student_outputs.logits
    ce_loss=student_outputs.loss

    with torch.no_grad():
      teacher_outputs=self.teacher_model(**inputs)
      teacher_logits=teacher_outputs.logits

    T=self.temperature
    student_log_probs=F.log_softmax(student_logits/T,dim=-1)
    teacher_probs=F.softmax(teacher_logits/T,dim=-1)

    kl_loss=F.kl_div(student_log_probs,teacher_probs,reduction='none')
    kl_loss=kl_loss.sum(dim=-1)

    mask=labels!=-100
    kl_loss=(kl_loss*mask).sum()/mask.sum()
    kl_loss=kl_loss*(T**2)
    loss=self.alpha*ce_loss+(1-self.alpha)*kl_loss
    return (loss,student_outputs) if return_outputs else loss

In [ ]:
trainer=DistillationSeq2SeqTrainer(
    teacher_model=teacher_model,
    temperature=2.0,
    alpha=0.7,
    model=student_model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    data_collator=data_collator,
    processing_class=student_tokenizer
)

In [ ]:
trainer.train()

Step,Training Loss
50,99.538662
100,71.402383
150,54.217881
200,35.973572
250,27.290337
300,21.785471
350,18.955657
400,17.292670
450,15.861276
500,15.149453


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=891, training_loss=27.090188451353814, metrics={'train_runtime': 1639.8933, 'train_samples_per_second': 8.69, 'train_steps_per_second': 0.543, 'total_flos': 7018278470184960.0, 'train_loss': 27.090188451353814, 'epoch': 3.0})

In [ ]:
def summarize_text(text):
    student_model.eval()

    input_text = prefix + text

    inputs = student_tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():
        summary_ids = student_model.generate(
            **inputs,
            max_length=128,
            min_length=15,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    return student_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

In [ ]:
my_text = """
Son illərdə süni intellekt texnologiyaları səhiyyə sahəsində daha geniş tətbiq olunmağa başlayıb.
Xəstəxanalarda tibbi görüntülərin analizi, xəstəliklərin erkən aşkarlanması və pasiyent məlumatlarının
emalı üçün müxtəlif AI modellərindən istifadə edilir. Bu texnologiyalar həkimlərə daha sürətli qərar
verməyə kömək etsə də, mütəxəssislər insan nəzarətinin vacib olduğunu bildirirlər. Çünki səhiyyədə
verilən qərarlar birbaşa insan həyatı ilə bağlıdır və modelin səhvləri ciddi nəticələrə səbəb ola bilər.
Buna görə də süni intellekt həkimi əvəz edən sistem kimi deyil, həkimə kömək edən alət kimi
qiymətləndirilməlidir.
"""

print(summarize_text(my_text))

<extra_id_0> texnologiyaları səhiyyə sahəsində daha geniş tətbiq olunub


In [ ]:
trainer.save_model("./distilled-mt5-az-summary-final")
student_tokenizer.save_pretrained("./distilled-mt5-az-summary-final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./distilled-mt5-az-summary-final/tokenizer_config.json',
 './distilled-mt5-az-summary-final/tokenizer.json')